# ST7 Project 2026

## Algorithm
1. **[LONG]** Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. **[LONG]** Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [ ]:
import numpy as np
import h5py
import os
import subprocess

# pysem toolkit
from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from pysem.parse_sem3d_snapshots import ParseSEM3DSnapshots
from pysem.generate_h5_materials import write_h5

## 1. Paths and parameters

In [ ]:
# --- Paths ---
# WKDIR: working directory where SEM3D is launched
# must contain: input.spec, material.spec, stations.txt, gaussian_stf.txt, sem/mesh4spec.*.h5

# D_OBS_DIR: directory containing observed data (tutorial2/prot/Protection_.../Capteurs/)

# --- Initial material parameters m_0 ---
# domain limits, discretization, initial gradient

# --- Optimization parameters ---
# N_ITER: maximum number of RTM iterations
# TOL: gradient convergence tolerance
# ALPHA_0: initial step size for line search
# ARMIJO_C: backtracking reduction constant
# ARMIJO_TAU: sufficient decrease factor

## 2. Load observed data d_obs

In [ ]:
# Read observed traces (tutorial2) using ParseSEM3DH5Traces
# d_obs shape: (N_receivers, 3, N_timestep)
# components: 0=x, 1=y, 2=z

## 3. Initial material m_0

In [ ]:
# Generate initial material HDF5 files (e.g. homogeneous model)
# output: example_la.h5, example_mu.h5, example_ds.h5 in WKDIR

# m_la: 3D array of Lambda
# m_mu: 3D array of Mu
# m_ds: 3D array of Rho (fixed, not inverted)

## 4. Algorithm

In [ ]:
# CG variable initialization
# p_prev = 0  (previous direction)
# g_prev = 0  (previous gradient)
# J_prev = inf

# for n in range(N_ITER):

    # ── STEP 1 ────────────────────────────────────────
    # forward_solve(WKDIR, m_la, m_mu)
    # Launch SEM3D with current material
    # Wait for SLURM job completion
    # Expected output:
    #   - WKDIR/prot/Protection_XXXXXX/Capteurs/capteurs.*.h5  (u_sim at receivers)
    #   - WKDIR/res/Rsem*/sem_field.*.h5                        (u(x,t) snapshots)

    # ── STEP 2: MISFIT ────────────────────────────────────────────────────
    # Read u_sim from capteurs with ParseSEM3DH5Traces
    # Compute residuals: r = u_sim - d_obs  (shape: N_rec x 3 x N_t)
    # Compute cost:      J = 0.5 * ||r||^2
    # Print J to monitor convergence

    # ── STEP 3 ────────────────────────────────────────
    # adjoint_solve(WKDIR, r)
    # Time-reverse r: r_adj(t) = r(T - t)
    # Inject r_adj as sources at receivers in input.spec
    # Launch SEM3D backward
    # Expected output:
    #   - WKDIR/res_adj/Rsem*/sem_field.*.h5  (Λ(x,t) snapshots)

    # ── STEP 4: GRADIENT ─────────────────────────────────────────────────
    # Read edev and evol of u(x,t) from forward snapshots with ParseSEM3DSnapshots
    # Read edev and evol of Λ(x,t) from adjoint snapshots with ParseSEM3DSnapshots
    # Integrate in time (sum over all timesteps):
    #   g_la(x) = sum_t  evol[u](x,t) * evol[Λ](x,t)   (gradient w.r.t. Lambda)
    #   g_mu(x) = sum_t  edev[u](x,t) : edev[Λ](x,t)   (gradient w.r.t. Mu)
    # Add regularization term if present

    # Convergence check
    # if ||g|| < TOL: break

    # ── STEP 5: CONJUGATE GRADIENT DIRECTION (Fletcher-Reeves) ────────────
    # if n == 0:
    #     p_la = -g_la
    #     p_mu = -g_mu
    # else:
    #     beta = (||g||^2) / (||g_prev||^2)
    #     p_la = -g_la + beta * p_la_prev
    #     p_mu = -g_mu + beta * p_mu_prev

    # ── STEP 6: LINE SEARCH (backtracking Armijo) ─────────────────────────
    # alpha = ALPHA_0
    # while True:
    #     m_la_trial = m_la + alpha * p_la
    #     m_mu_trial = m_mu + alpha * p_mu
    #     Write m_trial to HDF5
    #     Launch forward solve with m_trial  ← (BLACK BOX)
    #     Compute J_trial
    #     if J_trial < J + ARMIJO_TAU * alpha * (g · p): break
    #     alpha = ARMIJO_C * alpha

    # ── STEP 7: UPDATE ─────────────────────────────────────────────────────
    # m_la = m_la + alpha * p_la
    # m_mu = m_mu + alpha * p_mu
    # Write new material HDF5 files
    # Save current state (for restart)
    # p_la_prev, p_mu_prev = p_la, p_mu
    # g_prev = g